# ARC26 NVARC v3 — v2 + icecuber CPU program search merged into empty/weak attempts
NVARC 2025 (via the public LB33.89 perfpatch notebook) with: crc32 scoring seeds + PYTHONHASHSEED,
per-task exception handling, TTT deadline guard, cheap-first task order, identity fallback for empty outputs.
Pass 2 spends leftover time re-solving the lowest vote-margin tasks with new seeds; votes are pooled by score_kgmon.

In [ ]:
import os, time
rerun = os.getenv('KAGGLE_IS_COMPETITION_RERUN')
global_end_time = time.time() + (12 * 3600 - 600 if rerun else 6.0 * 3600 - 600)

In [ ]:
import os, glob
!ls /kaggle/input /kaggle/input/* | head -50
def _find(pattern):
    hits = sorted(glob.glob('/kaggle/input/**/' + pattern, recursive=True), key=len)
    return hits[0] if hits else None
f = _find('arc-agi_test_challenges.json')
if f: os.environ['ARC_COMP_DIR'] = os.path.dirname(f)
f = _find('qwen3_4b_grids15_sft139/**/config.json')
if f: os.environ['ARC_MODEL_DIR'] = os.path.dirname(f)
print('ARC_COMP_DIR', os.environ.get('ARC_COMP_DIR'), 'ARC_MODEL_DIR', os.environ.get('ARC_MODEL_DIR'))

In [ ]:
!pip uninstall -y tensorflow

In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import json
import pickle
import traceback
import zlib

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15

# Skip test-time training when less than this many seconds remain before end_time.
TTT_MIN_REMAINING = float(os.getenv("ARC_TTT_MIN_REMAINING", "900"))

# Pass 1 reproduces the NVARC baseline. Later passes re-solve uncertain tasks with fresh
# seeds; their candidates land in extra files so the decoder pools the votes.
ARC_PASS = int(os.getenv("ARC_PASS", "1"))
LORA_SEED = 42 if ARC_PASS == 1 else 42 + 1000 * ARC_PASS
TRAIN_AUG_SEED = 1 if ARC_PASS == 1 else 1 + 1000 * ARC_PASS
EVAL_AUG_SEED = 2 if ARC_PASS == 1 else 2 + 1000 * ARC_PASS
OUTPUT_SUFFIX = "" if ARC_PASS == 1 else f".p{ARC_PASS}"


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


# Minimal performance patch: preserve the baseline beam set and ranking, but transfer
# only the 12 ARC-token NLL values to CPU instead of every Qwen vocabulary logit.
_ARC_TOKEN_ID_CACHE = {}


def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    # Algebraically identical to: scores - logits.float().cpu().log_softmax(-1),
    # restricted to the same ARC_TOKENS used by the baseline DFS loop.
    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    # Keep logits on GPU and gather only the target-token scores. KV cache is not
    # consumed by teacher-forced scoring, so disabling it removes redundant writes.
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
    return result


# Sorted subkeys per test output: 0-1 id, 2-3 rot90, 4-5 rot180, 6-7 rot270,
# 8-9 T, 10-11 T.rot90, 12-13 T.rot180, 14-15 T.rot270 (two colour permutations each).
# Views in one group have the same grid shapes and therefore the same token length,
# which the patched decode attention (no padding mask) requires.
SAME_SHAPE_GROUPS = [(0, 1, 4, 5, 10, 11, 14, 15), (2, 3, 6, 7, 8, 9, 12, 13)]
# Above this prompt length decode 4 views at a time to bound KV-cache memory.
LONG_PROMPT_TOKENS = int(os.getenv("ARC_LONG_PROMPT_TOKENS", "6000"))


def make_view_batches(test_id_to_subkeys, eval_ds, formatter):
    """One 8-view decode batch per shape group and test output (4-view halves for long prompts)."""
    batches = []
    for group in SAME_SHAPE_GROUPS:
        for test_id, subkeys in test_id_to_subkeys.items():
            batch = [subkeys[i] for i in group if i < len(subkeys)]
            # Guard: cut_to_len can drop different train examples per view, so split by length.
            by_len = defaultdict(list)
            for subkey in batch:
                by_len[eval_ds.get_length(subkey, formatter, name="input")].append(subkey)
            for length, same_len in by_len.items():
                step = 4 if length > LONG_PROMPT_TOKENS else 8
                batches.extend(same_len[i:i + step] for i in range(0, len(same_len), step))
    return batches


def worker(rank, queue, end_time):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=LORA_SEED,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=LORA_SEED,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=os.environ.get("ARC_MODEL_DIR", "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"),
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = os.environ.get("ARC_COMP_DIR", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2") + "/arc-agi_test_challenges.json"
    else:
        test_path = os.environ.get("ARC_COMP_DIR", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2") + "/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break

        try:
            model = solve_task(rank, key, model, tokenizer, collator, formatter, default_weights, arc_test_set,
                               train_args, max_seq_length, max_new_tokens, max_score, dir_outputs, end_time)
        except Exception:
            print(f"[Rank {rank}] FAILED {key}\n{traceback.format_exc()}")
            gc.collect()
            torch.cuda.empty_cache()


def solve_task(rank, key, model, tokenizer, collator, formatter, default_weights, arc_test_set,
               train_args, max_seq_length, max_new_tokens, max_score, dir_outputs, end_time):
        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=TRAIN_AUG_SEED)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        # TTT has no internal time guard; near the deadline skip it and decode with the
        # freshly reset adapter (LoRA B=0, i.e. the base SFT model) so the run can't overrun.
        stats = None
        if end_time - time.time() < TTT_MIN_REMAINING and ARC_PASS > 1:
            # An untrained adapter's votes would only dilute pass 1's candidates.
            print(f"[Rank {rank}] skip {key} in pass {ARC_PASS}: {end_time - time.time():.0f}s left")
            return model
        if end_time - time.time() < TTT_MIN_REMAINING:
            print(f"[Rank {rank}] skip TTT for {key}: {end_time - time.time():.0f}s left")
        else:
            with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):

                trainer = UnslothFixedTrainer(
                    model=model,
                    tokenizer=tokenizer,
                    data_collator=collator,
                    train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                    dataset_text_field="text",
                    max_seq_length=max_seq_length,
                    args=UnslothTrainingArguments(**train_args),
                )

                stats = trainer.train()

                model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

                del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=2, seed=EVAL_AUG_SEED)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = make_view_batches(test_id_to_subkeys, eval_ds, formatter)

        with torch.inference_mode():
                
            known_scores = {}

            while batches:
                subkeys = batches.pop(0)

                spend_time = time.time() - start_time
                if spend_time > 1200 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                try:
                    dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)
                except torch.OutOfMemoryError:
                    if len(subkeys) == 1:
                        raise
                    # Retry the batch as two halves instead of losing the whole task.
                    print(f"[Rank {rank}] OOM decoding {len(subkeys)} views of {key}; splitting")
                    gc.collect()
                    torch.cuda.empty_cache()
                    half = len(subkeys) // 2
                    batches[:0] = [subkeys[:half], subkeys[half:]]
                    continue

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=zlib.crc32(bk.encode()) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey + OUTPUT_SUFFIX), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")

        # Kept outside dir_outputs: the decoder unpickles every file in that directory.
        with open(f"/kaggle/working/timing_rank{rank}.jsonl", "a") as f:
            f.write(json.dumps({"key": key, "seconds": round(spend_time, 1), "ttt": stats is not None, "pass": ARC_PASS}) + "\n")

        return model


In [ ]:
%%writefile starter.py
import bz2
import glob
import os
import pickle
import time
import json
import torch
import argparse
import torch.multiprocessing as mp
from collections import Counter, defaultdict

DEBUG_KEYS = ["0934a4d8", "36a08778", "981571dc", "aa4ec2a5"]
DIR_OUTPUTS = "/kaggle/inference_outputs"
# Start another pass only if at least this many seconds remain.
MIN_PASS_REMAINING = float(os.getenv("ARC_MIN_PASS_REMAINING", "1800"))
MAX_PASSES = int(os.getenv("ARC_MAX_PASSES", "2"))


def local_worker(rank, queue, end_time):

    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue
    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)

    from arc_solver import worker

    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")

    print(f"[Rank {rank}] start!")

    worker(rank, queue, end_time)

    print(f"[Rank {rank}] done!")


def estimated_work(task):
    # Rough proxy for TTT + decode cost: grid cells seen in training (x16 augments)
    # plus cells decoded per test output.
    train_cells = sum(len(p["input"]) * len(p["input"][0]) + len(p["output"]) * len(p["output"][0]) for p in task["train"])
    test_cells = sum(len(t["input"]) * len(t["input"][0]) for t in task["test"])
    return train_cells * 16 + test_cells * 8 * len(task["test"])


def vote_margin(n_outputs_by_key):
    """Per task: smallest (top-1 votes - top-2 votes) over its test outputs; -1 if an output has no candidate.

    A vote is one augmented view whose DFS produced that grid, as counted by score_kgmon.
    """
    votes = defaultdict(Counter)
    for path in glob.glob(os.path.join(DIR_OUTPUTS, "*")):
        with bz2.BZ2File(path) as f:
            for g in pickle.load(f):
                votes[os.path.basename(path).split(".")[0]][tuple(map(tuple, g["solution"]))] += 1
    margins = {}
    for key, n_outputs in n_outputs_by_key.items():
        per_output = []
        for i in range(n_outputs):
            top = [c for _, c in votes[f"{key}_{i}"].most_common(2)] + [0, 0]
            per_output.append(top[0] - top[1] if top[0] else -1)
        margins[key] = min(per_output)
    return margins


def run_pass(n_pass, keys, nprocs, end_time):
    os.environ["ARC_PASS"] = str(n_pass)
    # Start-up markers left by the previous pass would let every rank patch Unsloth at once.
    for marker in glob.glob("/kaggle/worker*"):
        os.remove(marker)
    print(f"*** pass {n_pass}: {len(keys)} tasks on {nprocs} GPUs, {end_time - time.time():.0f}s left")
    queue = mp.Manager().Queue()
    for key in keys:
        queue.put(key)
    for _ in range(nprocs):
        queue.put(None)
    mp.spawn(local_worker, args=(queue, end_time), nprocs=nprocs)


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = os.environ.get("ARC_COMP_DIR", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2") + "/arc-agi_test_challenges.json"
    else:
        test_path = os.environ.get("ARC_COMP_DIR", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2") + "/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    keys = list(data.keys())
    if not rerun_mode:
        # ARC_EVAL_KEYS: "debug" (4 tasks, default), "all", or comma-separated task ids.
        eval_keys = os.getenv("ARC_EVAL_KEYS", "debug")
        if eval_keys == "debug":
            keys = DEBUG_KEYS
        elif eval_keys != "all":
            keys = eval_keys.split(",")

    # Cheap tasks first: when time runs out, the tasks left undone are the expensive ones
    # instead of whatever sorts last alphabetically.
    keys = sorted(keys, key=lambda k: (estimated_work(data[k]), k))

    nprocs = torch.cuda.device_count() or 4

    run_pass(1, keys, nprocs, args.end_time)

    # Spend leftover time re-solving the least certain tasks first, with new seeds.
    for n_pass in range(2, MAX_PASSES + 1):
        if args.end_time - time.time() < MIN_PASS_REMAINING:
            break
        margins = vote_margin({k: len(data[k]["test"]) for k in keys})
        print(f"*** vote margins before pass {n_pass}: {sorted(Counter(margins.values()).items())}")
        order = sorted(keys, key=lambda k: (margins[k], estimated_work(data[k]), k))
        run_pass(n_pass, order, nprocs, args.end_time)


In [ ]:
import base64, io, os, subprocess, tarfile, time, json
CPU_DIR = "/tmp/cpu_solver"
cpu_proc = None
comp_dir = os.environ.get("ARC_COMP_DIR", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2")
cpu_challenges = f"{comp_dir}/arc-agi_test_challenges.json" if os.getenv("KAGGLE_IS_COMPETITION_RERUN") \
    else f"{comp_dir}/arc-agi_evaluation_challenges.json"
try:
    os.makedirs(CPU_DIR, exist_ok=True)
    tarfile.open(fileobj=io.BytesIO(base64.b64decode("...")), mode="r:xz").extractall(CPU_DIR)
    print(subprocess.run("nproc; free -g | head -2; g++ --version | head -1", shell=True, capture_output=True, text=True).stdout)
    t0 = time.time()
    r = subprocess.run(["make", "-j8"], cwd=f"{CPU_DIR}/icecuber", capture_output=True, text=True)
    print("cpu solver build", r.returncode, f"{time.time()-t0:.0f}s", r.stderr[-2000:])
    if r.returncode == 0:
        cpu_proc = subprocess.Popen(
            ["python", f"{CPU_DIR}/run_icecuber.py", "--challenges", cpu_challenges, "--out", f"{CPU_DIR}/cpu.json",
             "--workers", "16", "--nice", "15", "--passes", "3:900:0,23:1200:0,33:1200:0", "--mem-mb", "6000",
             "--total-mem-mb", "64000", "--deadline", str(global_end_time - 1800)],
            stdout=open(f"{CPU_DIR}/cpu.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
except Exception as e:
    print("cpu solver disabled:", repr(e))


In [ ]:
import os
os.environ.setdefault('ARC_EVAL_KEYS', 'all')
!PYTHONHASHSEED=0 UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 python starter.py --end-time {global_end_time}

In [ ]:
%%writefile make_submission.py
import os
import json
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder

rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

comp_dir = os.environ.get("ARC_COMP_DIR", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2")
if rerun_mode:
    data = ArcDataset.from_file(f"{comp_dir}/arc-agi_test_challenges.json")
else:
    data = ArcDataset.from_file(f"{comp_dir}/arc-agi_evaluation_challenges.json")
    data = data.load_replies(f"{comp_dir}/arc-agi_evaluation_solutions.json")

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)

decoder.load_decoded_results("/kaggle/inference_outputs")

submission = data.get_submission(decoder.run_selection_algo())

# Outputs with no candidate at all: try the test input itself as attempt_1 instead of [[0]].
n_empty = 0
for key, outputs in submission.items():
    for i, attempts in enumerate(outputs):
        if attempts["attempt_1"] == [[0]]:
            attempts["attempt_1"] = data.queries[key]["test"][i]["input"]
            n_empty += 1
print(f"*** outputs without candidates: {n_empty} / {sum(map(len, submission.values()))}")

with open("submission.json", "w") as f:
    json.dump(submission, f)

if not rerun_mode:
    decoder.benchmark_selection_algos()

    # Paired check of later passes: same tasks, pass-1 candidates only vs all passes pooled.
    import re, shutil, tempfile
    with tempfile.TemporaryDirectory() as tmp:
        for name in os.listdir("/kaggle/inference_outputs"):
            if not re.fullmatch(r"p\d+", name.rsplit(".", 1)[-1]):
                shutil.copy(os.path.join("/kaggle/inference_outputs", name), tmp)
        pass1 = ArcDecoder(data.split_multi_replies(), n_guesses=2)
        pass1.load_decoded_results(tmp)
        pass1_submission = data.get_submission(pass1.run_selection_algo())
    print("*** Pass-1-only score:", data.validate_submission(pass1_submission))
    with open("submission.json", "r") as f:
        reload_submission = json.load(f)
    print("*** Reload score:", data.validate_submission(reload_submission))

    # Keep the raw candidates for offline selection/re-scoring work (eval commit runs only).
    import tarfile
    with tarfile.open("/kaggle/working/inference_outputs.tar", "w") as tar:
        tar.add("/kaggle/inference_outputs", arcname="inference_outputs")
    print("*** saved inference_outputs.tar:", os.path.getsize("/kaggle/working/inference_outputs.tar") // 1024, "KB")


In [ ]:
!PYTHONHASHSEED=0 python make_submission.py

In [ ]:
import os, subprocess, signal, time, json, shutil
def _score(sub, sol):
    tot = 0.0
    for k, outs in sol.items():
        att = sub.get(k, [])
        tot += sum(i < len(att) and g in (att[i].get("attempt_1"), att[i].get("attempt_2")) for i, g in enumerate(outs)) / len(outs)
    return tot
try:
    if cpu_proc is not None:
        # merge even if passes are still going: kill, then re-run the driver with no passes to merge what exists
        if cpu_proc.poll() is None:
            os.killpg(cpu_proc.pid, signal.SIGKILL); time.sleep(1)
        subprocess.run(["python", f"{CPU_DIR}/run_icecuber.py", "--challenges", cpu_challenges,
                        "--out", f"{CPU_DIR}/cpu.json", "--passes", ""], check=False)
        print(open(f"{CPU_DIR}/cpu.log").read()[-3000:])
    if cpu_proc is not None and os.path.exists(f"{CPU_DIR}/cpu.json"):
        rerun = os.getenv("KAGGLE_IS_COMPETITION_RERUN")
        if not rerun:
            shutil.copy("submission.json", "/kaggle/working/submission_nvarc.json")
            shutil.copy(f"{CPU_DIR}/cpu.json", "/kaggle/working/cpu.json")
            shutil.copy(f"{CPU_DIR}/cpu.json.stats.jsonl", "/kaggle/working/cpu_stats.jsonl")
        subprocess.run(["python", f"{CPU_DIR}/merge_cpu.py", "--submission", "submission.json",
                        "--cpu", f"{CPU_DIR}/cpu.json", "--challenges", cpu_challenges,
                        "--out", "submission_merged.json"], check=True)
        merged = json.load(open("submission_merged.json"))
        assert set(merged) == set(json.load(open("submission.json")))
        if not rerun:
            sol = json.load(open(cpu_challenges.replace("_challenges", "_solutions")))
            print("*** score NVARC only:", _score(json.load(open("submission.json")), sol), " merged:", _score(merged, sol))
        shutil.move("submission_merged.json", "submission.json")
except Exception as e:
    print("cpu merge skipped:", repr(e))
